In [1]:
import pandas as pd
import numpy as np

hotel_master = pd.read_csv(
    "../data/processed/hotel_master.csv"
)

destination_master = pd.read_csv(
    "../data/processed/destination_master.csv"
)

print("Hotel master shape:", hotel_master.shape)
print("Destination master shape:", destination_master.shape)

display(hotel_master.head())

Hotel master shape: (2130, 11)
Destination master shape: (76, 12)


,Type,Name,Address,Rooms,Grade,District,AGA Division,PS/MC/UC,Logitiute,Latitude,District_Key
0,Boutique Hotels,THE THEVA RESIDENCY,"11/B5/10-1 06TH LANE, HANTANA,KANDY",10,UNKNOWN,Kandy,Kandy Divisional Secretariat,Kandy,80.635411,7.276036,kandy
1,Boutique Hotels,HIGHLAND VILLA,"351, ABIMANGAMA ROAD, KUMBALGAMA, WELIGAMA",10,UNKNOWN,Matara,Weligama Divisional Secretariat,Weligama Pradeshiya Sabha,80.409972,5.960334,matara
2,Boutique Hotels,ULAGALLA WALAWWA RESORT,"THIRAPPANE, ANURADHAPURA",21,UNKNOWN,Anuradhapura,Anuradhapura East,Anuradhapura,80.545063,8.205927,anuradhapura
3,Boutique Hotels,GALLE FORT HOTEL,"NO.28, CHURCH STREET, FORT , GALLE",14,UNKNOWN,Galle,Galle Divisional Secretariat,Galle,80.217563,6.026649,galle
4,Boutique Hotels,THE ELEPHANT CORRIDOR,"POTHANA,KIBISSA, SIGIRIYA",21,UNKNOWN,Matale,UNKNOWN,Matale,80.710743,7.943525,matale


In [2]:

def recommend_hotels(destination_name, accommodation_type=None, top_n=5):
    # Find the selected destination
    destination = destination_master[
        destination_master["Location_Name"].str.casefold()
        == destination_name.strip().casefold()
    ]

    if destination.empty:
        return pd.DataFrame()

    # Get the destination's mapped district
    district = destination.iloc[0]["Mapped_District"]

    if pd.isna(district) or str(district).strip() == "":
        print("District mapping is unavailable for this destination.")
        return pd.DataFrame()

    # Filter accommodation by district
    results = hotel_master[
        hotel_master["District_Key"]
        == str(district).strip().casefold()
    ].copy()

    # Optionally filter by accommodation type
    if accommodation_type:
        results = results[
            results["Type"].str.casefold()
            == accommodation_type.strip().casefold()
        ]

    # Remove duplicate accommodation names
    results = results.drop_duplicates(subset=["Name"])

    # Return available information only
    columns = [
        "Name",
        "Type",
        "District",
        "Grade",
        "Rooms",
        "Address"
    ]

    return results[columns].head(top_n).reset_index(drop=True)


In [3]:
bentota_hotels = recommend_hotels(
    destination_name="Bentota Beach",
    top_n=5
)

print("Number of accommodation options:", len(bentota_hotels))
display(bentota_hotels)

Number of accommodation options: 5


,Name,Type,District,Grade,Rooms,Address
0,GALLE FORT HOTEL,Boutique Hotels,Galle,UNKNOWN,14,"NO.28, CHURCH STREET, FORT , GALLE"
1,ADITYA,Boutique Hotels,Galle,UNKNOWN,12,"719/1, GALLE ROAD, DEVENIGODA,RATHGAMA"
2,CLUB VILLA,Boutique Hotels,Galle,UNKNOWN,15,"NO.138/15, GALLE ROAD, BENTOTA"
3,SAMAN VILLAS,Boutique Hotels,Galle,UNKNOWN,27,"ATURUWELLA, INDURUWA"
4,TAMARIND HILL HOTEL,Boutique Hotels,Galle,UNKNOWN,11,"NO.288,GALLE ROAD, DADELLA, GALLE"


In [4]:

boutique_hotels = recommend_hotels(
    destination_name="Bentota Beach",
    accommodation_type="Boutique Hotels",
    top_n=5
)

print("Boutique hotels found:", len(boutique_hotels))
display(boutique_hotels)


Boutique hotels found: 5


,Name,Type,District,Grade,Rooms,Address
0,GALLE FORT HOTEL,Boutique Hotels,Galle,UNKNOWN,14,"NO.28, CHURCH STREET, FORT , GALLE"
1,ADITYA,Boutique Hotels,Galle,UNKNOWN,12,"719/1, GALLE ROAD, DEVENIGODA,RATHGAMA"
2,CLUB VILLA,Boutique Hotels,Galle,UNKNOWN,15,"NO.138/15, GALLE ROAD, BENTOTA"
3,SAMAN VILLAS,Boutique Hotels,Galle,UNKNOWN,27,"ATURUWELLA, INDURUWA"
4,TAMARIND HILL HOTEL,Boutique Hotels,Galle,UNKNOWN,11,"NO.288,GALLE ROAD, DADELLA, GALLE"


In [5]:

print(
    "All results match selected type:",
    boutique_hotels["Type"].str.casefold().eq("boutique hotels").all()
)

print(
    "Duplicate names:",
    boutique_hotels["Name"].duplicated().sum()
)

print(
    "Missing accommodation names:",
    boutique_hotels["Name"].isna().sum()
)


All results match selected type: True
Duplicate names: 0
Missing accommodation names: 0


In [6]:
bentota_hotels.to_csv(
    "../data/processed/bentota_hotel_recommendations_test.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Hotel recommendation test saved successfully!")

Hotel recommendation test saved successfully!
